## Notebook 02 — Analyse des données


### <u> 1. Étapes du notebook </u>

1. **Découpage** — mise de côté du jeu de test
2. **Métadonnées et valeurs manquantes** — types, taux de manquants, interprétation
3. **Distributions et valeurs aberrantes** — échelles, outliers
4. **Étude des cibles** — équilibre du top 10, distribution de la position d'arrivée
5. **Corrélations** — lien des features avec la cible, redondances entre features
6. **Conclusions** — décisions pour la modélisation


<br/>

### <u> 2. Choix du découpage et de la validation </u>

**Train** — saisons 2006 à 2021 ; **Test** — saisons 2022 et après, mis de côté

**Découpage chronologique** — on prédit des saisons futures à partir du passé, comme en conditions réelles

**Validation croisée groupée par saison** — chaque fold contient des saisons entières, car les pilotes d'une même course ne sont pas indépendants

<br/>

### <u> 3. Résultat attendu </u>

- **Analyse sur le train uniquement** — la table n'est pas modifiée
- **Décisions** — imputation des valeurs manquantes, mise à l'échelle, features à garder

---------------------------------------------------------------------------------------------

In [ ]:
# ÉTAPE 1 : DÉCOUPAGE DES DONNÉES

# Import
import pandas as pd


# Chargement csv construit dans le Notebook 1
data = pd.read_csv("../data/processed-data/f1_table.csv")


# Découpage train (2006-2021) et test (2022 et +)
train = data[data["year"] <= 2021]
test = data[data["year"] >= 2022]


# Vérification du train
print(f"- Total de lignes du train : {len(train)} lignes")
print(f"- Le train s'étend de {train['year'].min()} à {train['year'].max()}") 
print(f"- Taux de top 10 : {train['top10'].mean():.3f}") # Taux du top 10
print(f"- Vérification du total: {len(train) + len(test)} = {len(data)}") # Aucune perte de données 
print()
print(f"5 premières lignes :\n{train.head()}")


# Constrcution de groupes par années 
groups_train = train["year"]


# Rappel des colonnes (NoteBook1)
IDENTIFIANTS = ["raceId", "driverId", "year", "round", "date", "driverRef"]
NUMERICAL_FEATURE = ["driver_avg_position_last5", "driver_dnf_rate_last10", "driver_season_points_before_race",
                     "driver_career_starts", "driver_avg_position_on_circuit",
                     "team_avg_position_last5", "team_dnf_rate_last10", "team_season_points_before_race",
                     "grid_position_clean", "grid_gap_to_teammate", "quali_gap_to_best_q1_percent", "quali_position"]
CATEGORIAL_FEATURE = ["constructorRef", "circuitRef"]
CIBLES = ["top10", "positionOrder"]

In [ ]:
# ÉTAPE 2 : MÉTADONNÉES ET VALEURS MANQUANTES

# Taille et types
print(f"Dimensions : {train.shape}")
train.info()


# Features numériques : statistiques 
display(train[NUMERICAL_FEATURE].describe())


# Features catégorielles : valeurs différentes et répartition
display(train[CATEGORIAL_FEATURE].nunique())
display(train["constructorRef"].value_counts())
display(train["circuitRef"].value_counts())


# Valeurs manquantes : taux par colonne (uniquement les colonnes concernées)
manquants = train.isna().mean().round(3)
display(manquants[manquants > 0])

In [ ]:
# ÉTAPE 3 : DISTRIBUTIONS ET VALEURS ABERRANTES

# Imports 
import matplotlib.pyplot as plt


# Histogramme valeurs numériques 
train[NUMERICAL_FEATURE].hist(bins=30, figsize=(16, 12))
plt.show()

In [ ]:
# ÉTAPE 4 : ÉTUDE DES CIBLES

# Cible 1 : top 10 (0 pour NON et 1 pour OUI)
# Proportion de chacune des deux possibiltés
display(train["top10"].value_counts(normalize=True))


# Cible 2 : position d'arrivée (distribution)
# Histogramme de la position d'arrivée 
train["positionOrder"].hist(bins=24)
plt.show()

In [ ]:
# ÉTAPE 5 : CORRÉLATIONS

# Imports
import seaborn as sns

# Lien de chaque feature avec la position d'arrivée (Corrélation de Pearson)
correlations = train[NUMERICAL_FEATURE + ["positionOrder"]].corr()["positionOrder"].drop("positionOrder")
display(correlations.sort_values())


# Redondances entre features (heatmap)
plt.figure(figsize=(12, 9))
sns.heatmap(train[NUMERICAL_FEATURE].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.show()


# Taux de top 10 selon la position sur la grille
train.groupby("grid_position_clean")["top10"].mean().plot(kind="bar")
plt.show()

## Étape 6 — Conclusions


### <u> 1. Découpage </u>

- **Train (6 555 lignes)** — saisons 2006 à 2021
- **Test (1 359 lignes, soit 17 %)** — saisons 2022 et après


<br/>

### <u> 2. Valeurs manquantes </u>

- **Position moyenne sur ce circuit dans le passé (24 % de manquants)** — le pilote court pour la première fois sur le circuit 


- **Taux d'abandons du pilote sur les courses passées (1 % de manquants)** — c'est la première course de sa carrière


- **Taux d'abandon de l'écurie sur les courses passées (0,6 % de manquants)** — c'est la première course de l'écurie


- **Écart entre le temps en Q1 du pilote et le meilleur temps en Q1 (1,3 % de manquants)** — le pilote n'a pas réalisé de temps en Q1 (accident, panne, exclusion)


**Conclusion** : ces valeurs manquantent ne sont pas des erreurs ; elles doivent être remplacées par la médiane de la colonne dans le pipeline

<br/>

### <u> 3. Distributions et valeurs extrêmes </u>

- **Échelles très différentes** — les taux d'abandon vont de 0 à 1, alors que les points de l'écurie montent jusqu'à 722 : sans correction, certains modèles donneraient plus de poids aux grands nombres

- **Points de la saison très asymétriques** — la plupart des pilotes ont peu de points (médiane de 12), quelques-uns en ont énormément (jusqu'à 387) ; de plus, le barème a changé en 2010 (25 points par victoire au lieu de 10), donc les points ne se comparent pas d'une saison à l'autre

- **Valeurs extrêmes** — départs des stands (grille = 25) et qualifications ratées (écart plafonné à 10 %)

**Décision** : mise à l'échelle (StandardScaler) pour la régression logistique, le k-NN et le MLP, qui sont sensibles aux écarts d'échelle ; inutile pour les arbres ; valeurs extrêmes conservées, car elles se reproduiront dans le test

<br/>

### <u> 4. Features catégorielles </u>

- **30 écuries** — plusieurs sont en réalité la même équipe sous des noms différents (Toro Rosso → AlphaTauri, Force India → Racing Point etc ...)

- **36 circuits** — certains n'apparaissent qu'une seule saison dans le train (Jeddah, Losail, Zandvoort)

- **Catégories absentes du train** — l'écurie rb et les circuits miami et vegas n'existent qu'à partir de 2022 : le modèle ne les aura jamais vus pendant l'entraînement

**Décision** : encodage one-hot avec (handle_unknown="ignore"), pour que le modèle ne plante pas face à une écurie ou un circuit inconnu


<br/>

### <u> 5. Cibles </u>

- **Top 10 (46,8 % de oui)** — les deux classes sont presque aussi fréquentes l'une que l'autre : à partir d'une certaine date il n'y a plus 20 pilotes par course mais 22 puis 24

- **Position d'arrivée** — chaque position de 1 à 20 apparaît exactement une fois par course, d'où une distribution plate ; les positions 21 à 24 sont plus rares, car seules certaines saisons avaient plus de 20 voitures

**Décision** : pas de rééchantillonnage ; top 10 évalué par ROC AUC, F1 et accuracy ; position évaluée par la MAE (erreur moyenne en nombre de places)

<br/>

### <u> 6. Corrélations </u>

- **Features les plus liées au résultat** — position en qualification (0,63), sur la grille (0,61) et forme récente de l'écurie (0,59) : partir devant et rouler dans une écurie en forme est ce qui prédit le mieux l'arrivée

- **Features redondantes** — grille et qualification (0,96), points du pilote et de l'écurie (0,96), forme du pilote et de l'écurie (0,91) : chaque paire apporte presque la même information

- **Top 10 selon la grille** — 90 % des pilotes partant en pole finissent dans le top 10, 59 % depuis la 10e place, 46 % depuis la 11e

**Décision** : les modèles linéaires seront régularisés, car des features redondantes rendent leurs coefficients instables ; retirer une feature de chaque paire sera testé ensuite pour mesurer l'effet

<br/>


### <u> 7. Récapitulatif des décisions </u>

- **Valeurs manquantes** — imputation par la médiane (SimpleImputer)
- **Mise à l'échelle** — StandardScaler pour les modèles linéaires, k-NN et MLP ; aucune pour les arbres
- **Valeurs extrêmes** — conservées
- **Catégorielles** — OneHotEncoder(handle_unknown="ignore")
- **Rééchantillonnage** — aucun
- **Métriques** — ROC AUC, F1, accuracy (top 10) ; MAE (position)
- **Modèles linéaires** — régularisés
- **Features redondantes** — tester leur suppression
---------------------------------------------------------------------------------------------